# Day 046 — Exercise 5: TrendAnalyzer

**What you'll build:** The `TrendAnalyzer` class — `load(df, date_col, value_col)` (fluent, returns self) and `summary() -> dict` — combining parse, resample, rolling, and change analysis into one callable object.

**Why it matters:** The four functions you built in Exercises 1-4 are useful individually, but an analyst needs them composed. `TrendAnalyzer` is that composition — one `.load(...).summary()` call gives you everything: period count, start/end date, total change, trend direction, and pre-computed rolling stats.

## Provided: All Helper Functions

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def make_sample_ts(n_days: int = 90, seed: int = 42) -> pd.DataFrame:
    """Return a reproducible daily time-series DataFrame for exercises."""
    rng   = np.random.default_rng(seed)
    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')
    vals  = 1000.0 + (rng.standard_normal(n_days).cumsum() * 50)
    return pd.DataFrame({'date': dates.strftime('%Y-%m-%d'),
                         'value': vals.round(2)})


import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def parse_time_series(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df = df.set_index(date_col).sort_index()
    return df

def date_features(df: pd.DataFrame) -> pd.DataFrame:
    idx = df.index
    return pd.DataFrame({
        'year':        idx.year,
        'month':       idx.month,
        'day':         idx.day,
        'day_of_week': idx.dayofweek,
        'quarter':     idx.quarter,
    }, index=idx)


def resample_series(series: pd.Series, freq: str, agg: str = 'sum') -> pd.Series:
    return series.resample(freq).agg(agg)

def multi_freq_summary(series: pd.Series) -> dict:
    return {
        'daily':  resample_series(series, 'D'),
        'weekly': resample_series(series, 'W'),
    }


def rolling_mean(series: pd.Series, window: int, min_periods: int = 1) -> pd.Series:
    return series.rolling(window=window, min_periods=min_periods).mean()

def rolling_stats(series: pd.Series, window: int) -> pd.DataFrame:
    r = series.rolling(window=window, min_periods=1)
    return pd.DataFrame({
        'mean': r.mean(),
        'std':  r.std(),
        'min':  r.min(),
        'max':  r.max(),
    })


def period_changes(series: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        'value':      series,
        'change':     series.diff(),
        'pct_change': series.pct_change() * 100,
        'cumulative': series.cumsum(),
    })

## Your Implementation

In [ ]:
class TrendAnalyzer:
    """
    Compose time-series analysis into one callable object.

    Usage:
        ta = TrendAnalyzer()
        ta.load(df, 'date', 'value')
        report = ta.summary()
    """

    def __init__(self):
        # TODO: self._series = None
        pass

    def load(self, df: pd.DataFrame, date_col: str,
             value_col: str) -> 'TrendAnalyzer':
        """
        Parse the DataFrame, store the value column as self._series.
        Returns self for fluent chaining: ta.load(...).summary()
        """
        # TODO: self._series = parse_time_series(df, date_col)[value_col].dropna()
        # TODO: return self
        pass

    def summary(self) -> dict:
        """
        Return analysis dict with keys:
            n_periods, start_date, end_date,
            first_value, last_value,
            total_pct_change, trend_direction,
            weekly_avg, rolling_mean_7d, daily_changes
        """
        # TODO: s = self._series
        # TODO: if s is None or len(s) == 0: return {}
        # TODO: first, last = float(s.iloc[0]), float(s.iloc[-1])
        # TODO: total_pct = (last - first) / abs(first) * 100 if first != 0 else 0.0
        # TODO: direction = 'up' if total_pct > 1 else ('down' if total_pct < -1 else 'flat')
        # TODO: return {
        #     'n_periods': len(s),
        #     'start_date': str(s.index[0].date()),
        #     'end_date': str(s.index[-1].date()),
        #     'first_value': round(first, 2),
        #     'last_value': round(last, 2),
        #     'total_pct_change': round(total_pct, 2),
        #     'trend_direction': direction,
        #     'weekly_avg': resample_series(s, 'W', 'mean'),
        #     'rolling_mean_7d': rolling_stats(s, 7)['mean'],
        #     'daily_changes': period_changes(s),
        # }
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: class defined with load and summary methods
    try:
        assert 'TrendAnalyzer' in globals()
        for m in ('load', 'summary'):
            assert hasattr(TrendAnalyzer, m), f'missing method: {m}'
        passed += 1; print('\u2705 Check 1: TrendAnalyzer has load and summary')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: load() returns self (fluent)
    try:
        ta  = TrendAnalyzer()
        ret = ta.load(make_sample_ts(60), 'date', 'value')
        assert ret is ta, f'load() must return self, got {type(ret).__name__}'
        passed += 1; print('\u2705 Check 2: load() returns self (fluent chaining)')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: summary() returns dict with required keys
    try:
        report = ta.summary()
        assert isinstance(report, dict), \
            f'summary() should return dict, got {type(report).__name__}'
        required = ('n_periods', 'start_date', 'end_date',
                    'first_value', 'last_value', 'total_pct_change',
                    'trend_direction')
        for k in required:
            assert k in report, f'summary missing key: {k!r}'
        passed += 1; print('\u2705 Check 3: summary() returns dict with all required keys')
    except Exception as e:
        print(f'\u274c Check 3: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 4: n_periods is correct
    try:
        assert report['n_periods'] == 60, \
            f'n_periods should be 60, got {report["n_periods"]}'
        assert report['start_date'] == '2024-01-01', \
            f'start_date should be 2024-01-01, got {report["start_date"]}'
        passed += 1; print(f'\u2705 Check 4: n_periods=60, start_date=2024-01-01')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: trend_direction is one of 'up', 'down', 'flat'
    try:
        assert report['trend_direction'] in ('up', 'down', 'flat'), \
            f'trend_direction must be up/down/flat, got {report["trend_direction"]!r}'
        pct = report['total_pct_change']
        assert isinstance(pct, float), \
            f'total_pct_change should be float, got {type(pct).__name__}'
        passed += 1; print(f'\u2705 Check 5: trend_direction={report["trend_direction"]!r}, '
                           f'total_pct_change={pct:.2f}%')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class TrendAnalyzer:
    def __init__(self):
        self._series = None

    def load(self, df: pd.DataFrame, date_col: str, value_col: str) -> 'TrendAnalyzer':
        self._series = parse_time_series(df, date_col)[value_col].dropna()
        return self

    def summary(self) -> dict:
        s = self._series
        if s is None or len(s) == 0:
            return {}
        first, last = float(s.iloc[0]), float(s.iloc[-1])
        total_pct   = (last - first) / abs(first) * 100 if first != 0 else 0.0
        direction   = 'up' if total_pct > 1 else ('down' if total_pct < -1 else 'flat')
        return {
            'n_periods':        len(s),
            'start_date':       str(s.index[0].date()),
            'end_date':         str(s.index[-1].date()),
            'first_value':      round(first, 2),
            'last_value':       round(last, 2),
            'total_pct_change': round(total_pct, 2),
            'trend_direction':  direction,
            'weekly_avg':       resample_series(s, 'W', 'mean'),
            'rolling_mean_7d':  rolling_stats(s, 7)['mean'],
            'daily_changes':    period_changes(s),
        }
```

</details>